In [1]:
# Standard library
import os
import sys
import warnings

# Scientific computing
import numpy as np
import pandas as pd
import math
# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(color_codes=True)

# Single-cell & image analysis
import anndata as ad
import scanpy as sc
import scimap as sm

# Interactive viewer
import napari
import pandas as pd
import os
import anndata as ad
# Ignore warnings for cleaner output
warnings.filterwarnings('ignore')
import scanpy as sc

Running SCIMAP  2.3.5


/opt/anaconda3/envs/scimap_sep17/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html



In [3]:
import os
import pandas as pd
import anndata as ad
import numpy as np

markers = ["DAPI_0", "488_background", "555_background", "647_background", "750_background",
           "DAPI_1", "CyclinE_1", "p27_1", "PanCK_1",
           "DAPI_2", "PCNA_2", "CyclinD1_2", "p21_2", "PH3_2",
           "DAPI_3", "Ki-67_3", "Geminin_3", "CyclinB1_failed", "Vimentin_3",
           "DAPI_4", "P-RPA32_4", "Jun_4", "HLA-DPB1_4", "pRb_4",
           "DAPI_5", "LaminB1_5", "PAX8_5", "H2AX_5", "bCatenin_failed",
           "DAPI_6", "N-Cadherin_6", "aSMA_6", "CyclinB1_6", "Iba1_6",
           "DAPI_7", "E-Cadherin_7", "CD3D_7", "Zic2_7", "bCatenin_7"]

tribus_run_path = r"/Volumes/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Analysis/01-phenotyping/tribus/labels_PanCK_0_aSMA_1_pRb_combined_PH3_gated_600"

adata_list = []

for filename in os.listdir(tribus_run_path):

    if "raw_tribus_annotated" not in filename:
        continue

    file_path = os.path.join(tribus_run_path, filename)
    df = pd.read_csv(file_path)

    # check missing markers
    missing = [m for m in markers if m not in df.columns]
    if missing:
        print(f"{filename} missing markers filled with NaN:", missing)

    # create expression matrix (missing markers become NaN)
    X = df.reindex(columns=markers)

    # observation metadata
    obs_data = df[["final_label", "X_centroid","Y_centroid","CellID"]].copy()

    # add missing metadata columns
    for col in ["quartile", "dcc_filename_1", "dcc_filename_2", "tumor_proportion", "geomx_roi"]:
        obs_data[col] = np.nan

    # image id from filename
    image_id = "_".join(os.path.splitext(filename)[0].split("_")[:3])
    obs_data["imageid"] = image_id

    # make unique cell IDs
    obs_data.index = [f"{image_id}_cell{i}" for i in range(len(obs_data))]

    # marker metadata
    var_data = pd.DataFrame(index=markers)

    # create AnnData object
    adata = ad.AnnData(
        X=X.to_numpy(),
        obs=obs_data,
        var=var_data
    )

    adata_list.append(adata)

# combine datasets
adata = ad.concat(adata_list, join="outer", merge="same")

print(adata)

# define output path
output_file = os.path.join(tribus_run_path, "combined_adata_tribus_prb_combined_ph3_gated.h5ad")

# save as H5AD
adata.write(output_file)

print(f"Saved combined AnnData to {output_file}")

S032_iOvaL_b1_logic_table_proliferation_11_raw_tribus_annotated_pRb_combined_PH3_gated.csv missing markers filled with NaN: ['DAPI_0', '488_background', '555_background', '647_background', '750_background']
S091_iOme_logic_table_proliferation_11_raw_tribus_annotated_pRb_combined_PH3_gated.csv missing markers filled with NaN: ['DAPI_0', '488_background', '555_background', '647_background', '750_background']
AnnData object with n_obs × n_vars = 17321775 × 39
    obs: 'final_label', 'X_centroid', 'Y_centroid', 'CellID', 'quartile', 'dcc_filename_1', 'dcc_filename_2', 'tumor_proportion', 'geomx_roi', 'imageid'
Saved combined AnnData to /Volumes/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Analysis/01-phenotyping/tribus/labels_PanCK_0_aSMA_1_pRb_combined_PH3_gated_600/combined_adata_tribus_prb_combined_ph3_gated.h5ad


In [ ]:
import rerun as rr
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt 
import csv


from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union
from shapely.prepared import prep
from shapely.geometry import Point
import anndata as ad

In [2]:
adata_path="/home/ad/P-drive/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Analysis/01-phenotyping/tribus/labels_PanCK_0_aSMA_1_pRb_combined_PH3_gated_600/combined_adata_tribus_prb_combined_ph3_gated.h5ad"

adata=ad.read_h5ad(adata_path)

In [26]:
#CONVERTING ADATA OBJECT TO PANDAS DATAFRAME
df = pd.DataFrame(adata.X, columns=adata.var.index, index=adata.obs.index)
df = pd.concat([df, adata.obs], axis=1)

In [27]:
# Replace '_logic' in the entire column
df["imageid"] = df["imageid"].str.replace("_logic", "", regex=False)

# Optional: check the first few entries
print(df["imageid"].unique())

['S005_iOme' 'S015_iOme_b1' 'S015_iOme_b3' 'S026_iOme1' 'S027_iOme_b1'
 'S027_iOme_b3' 'S032_iOvaL_b3' 'S053_iOme' 'S057_iOme' 'S065_iOme'
 'S069_iAdnL' 'S072_iOme' 'S076_iOme' 'S080_iOme' 'S081_iOme' 'S083_iOme2'
 'S084_iOme_b1' 'S084_iOme_b3' 'S088_iOme' 'S095_iOme2' 'S098_iOme'
 'S100_iOme' 'S106_iOme' 'S107_iOme' 'S110_iOme' 'S112_iOme' 'S113_iOme'
 'S118_iOme' 'S120_iOme' 'S121_iOme' 'S123_iOme' 'S126_iOme' 'S128_iOme'
 'S130_iOme' 'S131_iOme' 'S139_iOme_b1' 'S139_iOme_b3' 'S188_iOme'
 'S189_iOme' 'S195_iOme' 'S197_iOme' 'S225_iOme' 'S229_iOme' 'S247_iOme'
 'S267_iOme' 'S268_iOme' 'S288_iOme' 'S309_iOme' 'S311_iOme' 'S332_iOme'
 'S333_iOvaR' 'S350_iOme' 'S355_iOme' 'S378_iOme' 'S380_iOme' 'S474_iOme'
 'S479_iOme' 'S482_iOme' 'S484_iOme' 'S493_iOme' 'S505_iAdnL' 'S527_iOme'
 'S535_iOme' 'S537_iOme' 'S032_iOvaL_b1' 'S091_iOme']


## Visualize with Rerun

In [14]:
sample = "S015_iOme_b3"#CHOOSE SAMPLE



subset_df = df[df["imageid"] == sample].reset_index(drop=True)

rr.init(f"{sample}", spawn=True)

labels = subset_df["final_label"].values
unique_labels = sorted(np.unique(labels))

cmap = plt.cm.get_cmap("Set1", len(unique_labels))
label_to_color = {
    label: np.array(cmap(i)[:3])
    for i, label in enumerate(unique_labels)
}
node_colors = np.array([label_to_color[label] for label in labels])

node_positions = (
    subset_df[["X_centroid", "Y_centroid"]]
    .to_numpy(dtype=np.float32)
)
cell_labels = subset_df["final_label"].astype(str).to_numpy()

rr.log(
    f"{sample}/cells",
    rr.Points2D(
        positions=node_positions,
        colors=(node_colors * 255).astype(np.uint8),
        radii=2, # size of points (cells)
        labels=cell_labels,   
    ),
)

/tmp/ipykernel_6992/2806428969.py:12: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("Set1", len(unique_labels))


## Add ROIs for viewing

In [16]:
import re


sample = "S015_iOme_b3"

ratio = 1 # == 1 if both cells and rois are in pixels

path = r"/home/ad/P-drive/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Transferred_ROIs"

# Read CSV as raw lines
with open(fr"{path}/{sample}.csv" ,"r") as f:
    lines = f.readlines()

# Header
header = lines[0].strip().split(",")
data_lines = lines[1:]

rows = []

for line in data_lines:
    # Split first column as ROI_Name
    roi_name, rest = line.strip().split(",", 1)

    # Use regex to find arrays inside brackets
    array_matches = re.findall(r"\[([^\]]+)\]", rest)

    if len(array_matches) != 2:
        raise ValueError(f"Expected 2 arrays in row: {line}")

    x_str, y_str = array_matches

    # Split by any whitespace or commas and convert to floats
    xs = [float(x) * ratio for x in re.split(r"[\s,]+", x_str.strip()) if x]
    ys = [float(y) * ratio for y in re.split(r"[\s,]+", y_str.strip()) if y]

    rows.append({"ROI_Name": roi_name, "X": xs, "Y": ys})

# Convert to DataFrame
roi_df = pd.DataFrame(rows)

for idx, row in roi_df.iterrows():
    roi_name = row["ROI_Name"]
    xs = row["X"]
    ys = row["Y"]

    pts = np.column_stack([xs, ys])

    rr.log(
        f"{sample}/rois/{roi_name}",
        rr.LineStrips2D([pts])
    )

## Add ROI information to cell data

In [ ]:
'''

files_list=['S032_iOme_b1', 'S380_iOme', 'S106_iOme', 'S120_iOme', 'S378_iOme', 
            'S311_iOme', 'S355_iOme', 'S080_iOme', 'S088_iOme', 'S118_iOme', 
            'S195_iOme', 'S091_iOme', 'S112_iOme', 'S247_iOme', 'S123_iOme',
              'S113_iOme', 'S015_iOme_b3', 'S084_iOme_b3', 'S139_iOme_b3',
                'S229_iOme', 'S130_iOme', 'S131_iOme', 'S107_iOme', 'S098_iOme',
                  'S069_iAdnL', 'S333_iOme', 'S100_iOme', 'S121_iOme', 'S083_iOme2', 
                  'S072_iOme', 'S057_iOme', 'S065_iOme', 'S076_iOme', 'S053_iOme',
                    'S032_iOvaL_b3', 'S309_iOme', 'S139_iOme_b1', 'S084_iOme_b1', 
                    'S015_iOme_b1', 'S027_iOme_b1', 'S197_iOme', 'S225_iOme']
for sample in files_list:

    df = df[df["imageid"] == sample].reset_index(drop=True)

    df["ROI"] = "None"


    ratio = 1 # == 1 if both cells and rois are in pixels

    path = r"/home/ad/P-drive/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Transferred_ROIs"

    rois_dict = dict()

    for sample in df["imageid"].unique():

        rois_dict[sample] = dict()

        # Read CSV as raw lines
        with open(fr"{path}/{sample}.csv" ,"r") as f:
            lines = f.readlines()

        # Header
        header = lines[0].strip().split(",")
        data_lines = lines[1:]

        rows = []

        for line in data_lines:
            # Split first column as ROI_Name
            roi_name, rest = line.strip().split(",", 1)

            # Use regex to find arrays inside brackets
            array_matches = re.findall(r"\[([^\]]+)\]", rest)

            if len(array_matches) != 2:
                raise ValueError(f"Expected 2 arrays in row: {line}")

            x_str, y_str = array_matches

            # Split by any whitespace or commas and convert to floats
            xs = [float(x) * ratio for x in re.split(r"[\s,]+", x_str.strip()) if x]
            ys = [float(y) * ratio for y in re.split(r"[\s,]+", y_str.strip()) if y]

            rows.append({"ROI_Name": roi_name, "X": xs, "Y": ys})

        # Convert to DataFrame
        roi_df = pd.DataFrame(rows)


        from shapely.geometry import Polygon
        from shapely.affinity import scale

        new_rows = []

        for _, row in roi_df.iterrows():
            xs = row["X"]
            ys = row["Y"]
            
            poly = Polygon(zip(xs, ys))
            rect = poly.minimum_rotated_rectangle
            
            if not rect.contains(poly):
                rect = scale(rect, xfact=1.001, yfact=1.001, origin="center")
            
            rect_coords = list(rect.exterior.coords)[:-1]
            
            new_rows.append({
                "ROI_Name": row["ROI_Name"],
                "X": [p[0] for p in rect_coords],
                "Y": [p[1] for p in rect_coords]
            })

        roi_df = pd.DataFrame(new_rows)


        roi_polygons = {}
        for idx, row in roi_df.iterrows():
            roi_name = row["ROI_Name"]
            xs = row["X"]
            ys = row["Y"]

            pts = np.column_stack([xs, ys])
            polygon = Polygon(pts)
            polygon = prep(polygon)
            rois_dict[sample][roi_name] = polygon

    for idx, cell in df.iterrows():
        sample = cell["imageid"]
        x = cell["X_centroid"]*ratio
        y = cell["Y_centroid"]*ratio
        point = Point(x, y)

        for roi_name, polygon in rois_dict[sample].items():
            if polygon.contains(point):
                df.at[idx, "ROI"] = roi_name
                break
df.to_csv("/home/ad/P-drive/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Transferred_ROIs/phenotyped_with_rois_transfered")
'''


In [ ]:
import os
import re
import numpy as np
import pandas as pd
from shapely.geometry import Polygon, Point
from shapely.affinity import scale
from shapely.prepared import prep

# List of sample IDs
files_list = ['S032_iOme_b1', 'S380_iOme', 'S106_iOme', 'S120_iOme', 'S378_iOme', 
              'S311_iOme', 'S355_iOme', 'S080_iOme', 'S088_iOme', 'S118_iOme', 
              'S195_iOme', 'S091_iOme', 'S112_iOme', 'S247_iOme', 'S123_iOme',
              'S113_iOme', 'S015_iOme_b3', 'S084_iOme_b3', 'S139_iOme_b3',
              'S229_iOme', 'S130_iOme', 'S131_iOme', 'S107_iOme', 'S098_iOme',
              'S069_iAdnL', 'S333_iOme', 'S100_iOme', 'S121_iOme', 'S083_iOme2', 
              'S072_iOme', 'S057_iOme', 'S065_iOme', 'S076_iOme', 'S053_iOme',
              'S032_iOvaL_b3', 'S309_iOme', 'S139_iOme_b1', 'S084_iOme_b1', 
              'S015_iOme_b1', 'S027_iOme_b1', 'S197_iOme', 'S225_iOme']

# Path to CSV files with ROI coordinated
path = r"/home/ad/P-drive/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Transferred_ROIs"

ratio = 1  # scaling factor for coordinates

# List to store processed DataFrames
all_samples_df = []

for sample in files_list:
    # Filter df for this sample
    sample_df = df[df["imageid"] == sample].copy().reset_index(drop=True)
    sample_df["ROI"] = "None"

    rois_dict = dict()

    # Read ROI CSV
    csv_path = os.path.join(path, f"{sample}.csv")
    if not os.path.exists(csv_path):
        print(f"CSV not found for {sample}, skipping...")
        continue

    with open(csv_path, "r") as f:
        lines = f.readlines()

    # Parse CSV
    header = lines[0].strip().split(",")
    data_lines = lines[1:]

    rows = []
    for line in data_lines:
        roi_name, rest = line.strip().split(",", 1)
        array_matches = re.findall(r"\[([^\]]+)\]", rest)
        if len(array_matches) != 2:
            raise ValueError(f"Expected 2 arrays in row: {line}")

        x_str, y_str = array_matches
        xs = [float(x) * ratio for x in re.split(r"[\s,]+", x_str.strip()) if x]
        ys = [float(y) * ratio for y in re.split(r"[\s,]+", y_str.strip()) if y]
        rows.append({"ROI_Name": roi_name, "X": xs, "Y": ys})

    roi_df = pd.DataFrame(rows)

    # Convert to minimum rotated rectangle
    new_rows = []
    for _, row_r in roi_df.iterrows():
        xs = row_r["X"]
        ys = row_r["Y"]
        poly = Polygon(zip(xs, ys))
        rect = poly.minimum_rotated_rectangle
        if not rect.contains(poly):
            rect = scale(rect, xfact=1.001, yfact=1.001, origin="center")
        rect_coords = list(rect.exterior.coords)[:-1]
        new_rows.append({
            "ROI_Name": row_r["ROI_Name"],
            "X": [p[0] for p in rect_coords],
            "Y": [p[1] for p in rect_coords]
        })
    roi_df = pd.DataFrame(new_rows)

    # Prepare polygons
    for _, row_r in roi_df.iterrows():
        roi_name = row_r["ROI_Name"]
        pts = np.column_stack([row_r["X"], row_r["Y"]])
        polygon = prep(Polygon(pts))
        rois_dict[roi_name] = polygon

    # Assign ROIs to cells
    for idx, cell in sample_df.iterrows():
        x = cell["X_centroid"] * ratio
        y = cell["Y_centroid"] * ratio
        point = Point(x, y)
        for roi_name, polygon in rois_dict.items():
            if polygon.contains(point):
                sample_df.at[idx, "ROI"] = roi_name
                break

    # Append this sample's DataFrame
    all_samples_df.append(sample_df)

# Concatenate all samples
final_df = pd.concat(all_samples_df, ignore_index=True)

# Save to CSV
final_df.to_csv(os.path.join(path, "phenotyped_with_rois_transferred.csv"), index=False)
print("All samples processed and saved.")

All samples processed and saved.


In [2]:
rois_path = "/home/ad/P-drive/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Transferred_ROIs/phenotyped_with_rois_transferred.csv"

dcc_path="/home/ad/P-drive/h30492/farkkilab2/9_EyeMT/Data/geomx/batch123/metadata/dcc_metadata_batch123_cleaned.csv"

In [ ]:
rois_transferred=pd.read_csv("/home/ad/P-drive/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Transferred_ROIs/phenotyped_with_rois_transferred.csv")
corresponding_dcc=pd.read_csv("/home/ad/P-drive/h30492/farkkilab2/9_EyeMT/Data/geomx/batch123/metadata/dcc_metadata_batch123_cleaned.csv")

/tmp/ipykernel_8159/2114248253.py:2: DtypeWarning: Columns (49) have mixed types. Specify dtype option on import or set low_memory=False.
  rois_transferred=pd.read_csv("/home/ad/P-drive/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Transferred_ROIs/phenotyped_with_rois_transferred.csv")


In [29]:
unique_pairs = corresponding_dcc[["Sample", "main_batch_nr"]].drop_duplicates()

sample_dict = unique_pairs.groupby("Sample")["main_batch_nr"].apply(list).to_dict()

print(sample_dict)

{'S015_iOme': [3], 'S015_pPer': [3], 'S015_post': [1], 'S015_pre': [1], 'S027_post': [1], 'S027_pre': [1], 'S032_iOval': [3], 'S032_pOme': [3], 'S032_post': [1], 'S032_pre': [1], 'S053_post': [1], 'S057_post': [1], 'S065_post': [1], 'S069_iAdnL': [2], 'S069_pOme': [2], 'S072_post': [1], 'S073_post': [1], 'S076_post': [1], 'S080_iOme2': [3], 'S081_iOme': [3], 'S083_iOme2': [2], 'S084_iOme': [3], 'S084_pAdn': [3], 'S084_post': [1], 'S084_pre': [1], 'S088_iOme': [3], 'S091_iOme1': [3], 'S098_iOme': [2], 'S100_iOme': [2], 'S106_iOme': [3], 'S107_iOme': [2], 'S112_iOme': [3], 'S113_iOme': [3], 'S118_iOme': [3], 'S120_iOme': [3], 'S121_iOme': [2], 'S123_iOme': [3], 'S130_iOme': [2], 'S131_iOme': [2], 'S139_iOme': [3], 'S139_pPer': [3], 'S139_post': [1], 'S139_pre': [1], 'S195_iOme1': [3], 'S197_iOme': [2], 'S225_iOme': [3], 'S229_iOme': [3], 'S229_pPer': [3], 'S247_iOme': [3], 'S268_iOme': [2], 'S309_iOme': [3], 'S311_iOme': [3], 'S333_iOvaR': [2], 'S333_pOme': [2], 'S355_iOme': [3], 'S378_i

In [30]:
rois_transferred["imageid"].unique()

array(['S380_iOme', 'S106_iOme', 'S120_iOme', 'S378_iOme', 'S311_iOme',
       'S355_iOme', 'S080_iOme', 'S088_iOme', 'S118_iOme', 'S195_iOme',
       'S091_iOme', 'S112_iOme', 'S247_iOme', 'S123_iOme', 'S113_iOme',
       'S015_iOme_b3', 'S084_iOme_b3', 'S139_iOme_b3', 'S229_iOme',
       'S130_iOme', 'S131_iOme', 'S107_iOme', 'S098_iOme', 'S069_iAdnL',
       'S100_iOme', 'S121_iOme', 'S083_iOme2', 'S072_iOme', 'S057_iOme',
       'S065_iOme', 'S076_iOme', 'S053_iOme', 'S032_iOvaL_b3',
       'S309_iOme', 'S139_iOme_b1', 'S084_iOme_b1', 'S015_iOme_b1',
       'S027_iOme_b1', 'S197_iOme', 'S225_iOme'], dtype=object)

In [21]:
corresponding_dcc.head()

,dcc_filename,Sample_ID,Slide_Name,Scan_Name,main_batch_nr,batch_nr,batch_nr_sample_collection,Sample,Patient,NACT_status,...,HRP_status,BRCA_status,PFS_quartile_b123,OS_quartile_b123,PFS_median_b123,OS_median_b123,PFS_quartile_paired,OS_quartile_paired,PFS_median_paired,OS_median_paired
0,DSP-1001660016606-G-A01.dcc,DSP-1001660016606-G-A01,No Template Control,NaN,1,1_8,1_8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2,2,NaN,NaN,NaN,NaN
1,DSP-1001660016606-G-A02.dcc,DSP-1001660016606-G-A02,NACT_TSI_2022_Batch8_13097-57209_140623,NACT_TSI_2022_Batch8_13097-57209_140623,1,1_8,1_8,S053_post,S053,post,...,1.0,0.0,1.0,1.0,1,1,NaN,NaN,NaN,NaN
2,DSP-1001660016606-G-A03.dcc,DSP-1001660016606-G-A03,NACT_TSI_2022_Batch8_13097-57209_140623,NACT_TSI_2022_Batch8_13097-57209_140623,1,1_8,1_8,S053_post,S053,post,...,1.0,0.0,1.0,1.0,1,1,NaN,NaN,NaN,NaN
3,DSP-1001660016606-G-A04.dcc,DSP-1001660016606-G-A04,NACT_TSI_2022_Batch8_13097-57209_140623,NACT_TSI_2022_Batch8_13097-57209_140623,1,1_8,1_8,S053_post,S053,post,...,1.0,0.0,1.0,1.0,1,1,NaN,NaN,NaN,NaN
4,DSP-1001660016606-G-A05.dcc,DSP-1001660016606-G-A05,NACT_TSI_2022_Batch8_13097-57209_140623,NACT_TSI_2022_Batch8_13097-57209_140623,1,1_8,1_8,S053_post,S053,post,...,1.0,0.0,1.0,1.0,1,1,NaN,NaN,NaN,NaN


In [20]:
corresponding_dcc["Roi_geomx"]

0           NaN
1       roi-001
2       roi-001
3       roi-002
4       roi-002
         ...   
1599     roi-78
1600     roi-79
1601     roi-79
1602     roi-80
1603     roi-81
Name: Roi_geomx, Length: 1604, dtype: object

## Matching of Samples in two tables

In [31]:
sample_batch_mapping = {
    "S015_iOme_b1": ("S015_iOme", 3),
    "S015_iOme_b3": ("S015_post", 1),
    "S027_iOme_b1": ("S027_post", 1),
    "S032_iOvaL_b3": ("S032_iOval", 3),
    "S053_iOme": ("S053_post", 1),
    "S057_iOme": ("S057_post", 1),
    "S065_iOme": ("S065_post", 1),
    #"S069_iAdnL": ("S069_iAdnL", 2),
    "S072_iOme": ("S072_post", 1),
    "S076_iOme": ("S076_post", 1),
    "S080_iOme": ("S080_iOme2", 3),
    "S083_iOme2": ("S083_iOme2", 2),
    "S084_iOme_b1": ("S084_post", 1),
    "S084_iOme_b3": ("S084_iOme", 3),
    "S088_iOme": ("S088_iOme", 3),
    "S091_iOme": ("S091_iOme1", 3),
    "S098_iOme": ("S098_iOme", 2),
    "S100_iOme": ("S100_iOme", 2),
    "S106_iOme": ("S106_iOme", 3),
    "S107_iOme": ("S107_iOme", 2),
    "S112_iOme": ("S112_iOme", 3),
    "S113_iOme": ("S113_iOme", 3),
    "S118_iOme": ("S118_iOme", 3),
    "S120_iOme": ("S120_iOme", 3),
    "S121_iOme": ("S121_iOme", 2),
    "S123_iOme": ("S123_iOme", 3),
    "S130_iOme": ("S130_iOme", 2),
    "S131_iOme": ("S131_iOme", 2),
    "S139_iOme_b1": ("S139_post", 1),
    "S139_iOme_b3": ("S139_iOme", 3),
    "S195_iOme": ("S195_iOme1", 3),
    "S197_iOme": ("S197_iOme", 2),
    "S225_iOme": ("S225_iOme", 3),
    "S229_iOme": ("S229_iOme", 3),
    "S247_iOme": ("S247_iOme", 3),
    "S309_iOme": ("S309_iOme", 3),
    "S311_iOme": ("S311_iOme", 3),
    "S355_iOme": ("S355_iOme", 3),
    "S378_iOme": ("S378_iOme", 3),
    "S380_iOme": ("S380_iOme", 3)
}

## Tumor DCC filenames transfer

In [ ]:
# Initialize the column
rois_transferred["dcc_filename_1"] = None

for idx, row in rois_transferred.iterrows():
    sample = row["imageid"]
    roi = row["ROI"]  # use the current row's ROI
    
    if sample in sample_batch_mapping:
        mapped_sample, batch = sample_batch_mapping[sample]
        
        # Correct boolean indexing
        matching_rows = corresponding_dcc[
            (corresponding_dcc["Sample"] == mapped_sample) &
            (corresponding_dcc["main_batch_nr"] == batch) &
            (corresponding_dcc["Roi_geomx_original"] == roi) &
            (corresponding_dcc["Segment"] == "tumor")
        ]
        
        # Assign dcc_filename if found
        if not matching_rows.empty:
            rois_transferred.loc[idx, "dcc_filename_1"] = matching_rows.iloc[0]["dcc_filename"]
            

In [42]:
rois_transferred_only_positive = rois_transferred[rois_transferred["ROI"].notna()]

In [44]:
# Initialize the column


for idx, row in rois_transferred_only_positive.iterrows():
    sample = row["imageid"]
    roi = row["ROI"]  # use the current row's ROI
    
    if sample in sample_batch_mapping:
        mapped_sample, batch = sample_batch_mapping[sample]
        
        # Correct boolean indexing
        matching_rows = corresponding_dcc[
            (corresponding_dcc["Sample"] == mapped_sample) &
            (corresponding_dcc["main_batch_nr"] == batch) &
            (corresponding_dcc["Roi_geomx_original"] == roi) &
            (corresponding_dcc["Segment"] == "tumor")
        ]
        
        # Assign dcc_filename if found
        if not matching_rows.empty:
            rois_transferred_only_positive.loc[idx, "dcc_filename_1"] = matching_rows.iloc[0]["dcc_filename"]

In [51]:
rois_transferred_only_positive_tumor=rois_transferred_only_positive[rois_transferred_only_positive["dcc_filename_1"].notna()]

In [ ]:
import pandas as pd

# Define numerator and denominator labels
numerator_labels = ["Tumor_Ki67", "Tumor_Ki67_pRb", "Tumor_PH3"]
denominator_labels = ["Tumor", "Tumor_Ki67_pRb", "Tumor_PH3", "Tumor_Ki67", "undefined_Tumor"]

# Calculate proportion per AOI
rois_transferred_only_positive_tumor['proportion_Ki67_pRb'] = (
    rois_transferred_only_positive_tumor
    .groupby('dcc_filename_1')['final_label']
    .transform(lambda x: x.isin(numerator_labels).sum() / max(x.isin(denominator_labels).sum(), 1))
)

# Count tumor cells per AOI
rois_transferred_only_positive_tumor["AOI_tumor_counts"] = (
    rois_transferred_only_positive_tumor
    .groupby('dcc_filename_1')['final_label']
    .transform(lambda x: x.isin(denominator_labels).sum())
)

print(rois_transferred_only_positive_tumor.head())

          DAPI_0  488_background  555_background  647_background  \
21   4283.564356      973.673267      212.623762      279.653465   
275  5349.956522     1092.200000      256.443478      284.026087   
314  3292.604010      852.699248      183.776942      276.661654   
469  3511.783505     1064.149485      242.448454      283.701031   
577  5823.947977     1018.407514      250.072254      281.985549   

     750_background       DAPI_1     CyclinE_1        p27_1     PanCK_1  \
21        80.495050  5973.435644   7966.841584  2777.366337  179.435644   
275       79.243478  6465.695652   4724.121739  1983.060870  159.226087   
314       80.796992  4371.804511   4360.746867  2827.017544  234.796992   
469       81.237113  5041.335052   8634.814433  3144.510309  454.623711   
577       80.728324  7117.335260  19234.476879  5079.031792  340.144509   

           DAPI_2  ...  CellID  quartile               dcc_filename_1  \
21    7986.277228  ...      29       NaN  DSP-1001660039812-B-F09.d

/tmp/ipykernel_8159/3613891905.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rois_transferred_only_positive_tumor['proportion_Ki67_pRb'] = (
/tmp/ipykernel_8159/3613891905.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rois_transferred_only_positive_tumor["AOI_tumor_counts"] = (


## Saving the cells with matched DCC_filenames

In [60]:
rois_transferred_only_positive_tumor.to_csv("/home/ad/P-drive/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Transferred_ROIs/cells_with_matched_dcc_files.csv")

## Thresholding the AOIs where the tumor cell counts are less than 30

In [1]:
rois_transferred_only_positive_tumor_thresholded=rois_transferred_only_positive_tumor[rois_transferred_only_positive_tumor["AOI_tumor_counts"]>30]

NameError: name 'rois_transferred_only_positive_tumor' is not defined

## Quartiles calculation

In [7]:
import pandas as pd 

rois_transferred_only_positive_tumor=pd.read_csv("/home/ad/P-drive/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Transferred_ROIs/cells_with_matched_dcc_files.csv")
rois_transferred_only_positive_tumor_thresholded=rois_transferred_only_positive_tumor[rois_transferred_only_positive_tumor["AOI_tumor_counts"]>30]

In [ ]:
# not the right way
rois_transferred_only_positive_tumor_thresholded["Ki67_pRb_quartile"] = pd.qcut(
    rois_transferred_only_positive_tumor_thresholded
        .drop_duplicates("dcc_filename_1")["proportion_Ki67_pRb"],
    4,
    labels=["Q1_low", "Q2", "Q3", "Q4_high"]
).reindex(rois_transferred_only_positive_tumor_thresholded.index, method="ffill")

/tmp/ipykernel_8159/584313082.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rois_transferred_only_positive_tumor_thresholded["Ki67_pRb_quartile"] = pd.qcut(


In [8]:
# 1. Get one value per sample (mean is typical)
sample_values = rois_transferred_only_positive_tumor_thresholded.groupby(
    "dcc_filename_1"
)["proportion_Ki67_pRb"].mean()

# 2. Compute quartiles
tertiles = pd.qcut(
    sample_values,
    3,
    labels=["T1_low", "T2_mid", "T3_high"]
)

# 3. Map back to all rows
rois_transferred_only_positive_tumor_thresholded["Ki67_pRb_tertile"] = \
    rois_transferred_only_positive_tumor_thresholded["dcc_filename_1"].map(tertiles)

/tmp/ipykernel_1705448/2551652000.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rois_transferred_only_positive_tumor_thresholded["Ki67_pRb_tertile"] = \


## Saving the file with quartiles calculated

In [9]:
rois_transferred_only_positive_tumor_thresholded.to_csv("/home/ad/P-drive/h30492/farkkilab2/4_CellCycle/tcycif/4_10_Expansion/Transferred_ROIs/cells_with_matched_dcc_files_qc_with_tertiles_1903.csv")